In [1]:
# 1.0 Call libraries
from llama_index.llms.ollama import Ollama
from llama_index.core import Settings


/home/ashok/langchain/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.3.0)/charset_normalizer (3.4.6) doesn't match a supported version!
  warnings.warn(


In [2]:
from llama_index.core import Settings
from llama_index.llms.ollama import Ollama
from llama_index.embeddings.ollama import OllamaEmbedding

llm = Ollama(model=   "llama3.2:3b-instruct-q8_0",  # "qwen3.5:latest",
             request_timeout=3600.0,
             temperature = 0.9
            )



# 4.1 Global LLM
Settings.llm = llm

# 4.2 Global Embedding Model
Settings.embed_model = OllamaEmbedding(model_name="nomic-embed-text")


/home/ashok/langchain/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
!mkdir -p 'data/'
!wget --user-agent "Mozilla" "https://arxiv.org/pdf/2307.09288.pdf" -O "data/llama2.pdf"

--2026-04-21 15:53:29--  https://arxiv.org/pdf/2307.09288.pdf
Resolving arxiv.org (arxiv.org)... 151.101.67.42, 151.101.195.42, 151.101.131.42, ...
Connecting to arxiv.org (arxiv.org)|151.101.67.42|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: /pdf/2307.09288 [following]
--2026-04-21 15:53:29--  https://arxiv.org/pdf/2307.09288
Reusing existing connection to arxiv.org:443.
HTTP request sent, awaiting response... 200 OK
Length: 13661300 (13M) [application/pdf]
Saving to: ‘data/llama2.pdf’

data/llama2.pdf     100%[===================>]  13.03M  8.88MB/s    in 1.5s    

2026-04-21 15:53:31 (8.88 MB/s) - ‘data/llama2.pdf’ saved [13661300/13661300]



In [4]:
from llama_index.core import SimpleDirectoryReader

documents = SimpleDirectoryReader("./data/").load_data()

In [7]:
from llama_index.core import VectorStoreIndex, StorageContext
from llama_index.core import Settings
from llama_index.vector_stores.qdrant import QdrantVectorStore
from qdrant_client import QdrantClient, AsyncQdrantClient

# creates a persistant index to disk
client = QdrantClient(host="localhost", port=6333)
aclient = AsyncQdrantClient(host="localhost", port=6333)

# create our vector store with hybrid indexing enabled
# batch_size controls how many nodes are encoded with sparse vectors at once
vector_store = QdrantVectorStore(
    "llama2_paper",
    client=client,
    aclient=aclient,
    enable_hybrid=True,
    fastembed_sparse_model="Qdrant/bm25",
    batch_size=20,
)

storage_context = StorageContext.from_defaults(vector_store=vector_store)
Settings.chunk_size = 512

index = VectorStoreIndex.from_documents(
    documents,
    storage_context=storage_context,
)

2026-04-21 15:56:05,708 - INFO - HTTP Request: GET http://localhost:6333/collections/llama2_paper/exists "HTTP/1.1 200 OK"
2026-04-21 15:56:05,709 - INFO - HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
2026-04-21 15:56:05,711 - INFO - HTTP Request: GET http://localhost:6333/collections/llama2_paper/exists "HTTP/1.1 200 OK"
2026-04-21 15:56:05,722 - INFO - HTTP Request: GET http://localhost:6333 "HTTP/1.1 200 OK"
Fetching 18 files: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████| 18/18 [00:01<00:00, 15.97it/s]
2026-04-21 15:56:07,906 - INFO - HTTP Request: GET http://localhost:6333/collections/llama2_paper/exists "HTTP/1.1 200 OK"
2026-04-21 15:56:12,431 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-04-21 15:56:15,922 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-04-21 15:56:19,340 - INFO - HTTP Request: POST http://localhost:11434/api/embed "H

In [8]:
query_engine = index.as_query_engine(
    similarity_top_k=2, sparse_top_k=12, vector_store_query_mode="hybrid"
)

2026-04-21 15:57:46,617 - INFO - HTTP Request: POST http://localhost:11434/api/show "HTTP/1.1 200 OK"


In [9]:
%%time

from IPython.display import display, Markdown

response = query_engine.query(
    "How was Llama2 specifically trained differently from Llama1?"
)

display(Markdown(str(response)))

2026-04-21 15:57:58,753 - INFO - HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-04-21 15:57:58,757 - INFO - HTTP Request: POST http://localhost:6333/collections/llama2_paper/points/query/batch "HTTP/1.1 200 OK"
2026-04-21 16:15:05,618 - INFO - HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


The training process incorporates safety fine-tuning techniques including supervised safety fine-tuning, safety RLHF, and safety context distillation. Additionally, the representation of different demographic groups in the pretraining data was analyzed by measuring rates of usage of demographic identity terms across specific axes such as religion, gender and sex, nationality, race and ethnicity, and sexual orientation.